# Quantile Forecasting (padronizado)


## 1) Importação do dataset e bibliotecas


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path("../../src").resolve()))
from setup import setup

CSV_PATH = "../../data/processed/financial_tools_datset.csv"
TARGET_COL = "Price"
HORIZON = 1
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15


## 2) Execução do `setup()` e alinhamento temporal


In [ ]:
raw_df = pd.read_csv(CSV_PATH)
raw_df["Date"] = pd.to_datetime(raw_df["Date"], format="%m/%d/%Y")
raw_df = raw_df.sort_values("Date").set_index("Date")

splits = setup(
    csv_path=CSV_PATH,
    target_col=TARGET_COL,
    horizon=HORIZON,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    save_artifacts=False,
)

processed_df = raw_df.reset_index().copy()
if "Change %" in processed_df.columns:
    processed_df["Change %"] = (
        processed_df["Change %"].str.replace("%", "", regex=False).astype(float) / 100
    )

processed_df = processed_df.sort_values("Date")
processed_df["target_return"] = (
    processed_df[TARGET_COL].pct_change(HORIZON).shift(-HORIZON)
)
processed_df = processed_df.dropna().set_index("Date")

n_rows = len(processed_df)
train_end = int(n_rows * TRAIN_RATIO)
val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))
val_index = processed_df.iloc[train_end:val_end].index

print("Validation length:", len(val_index))


## 3) Construção da ferramenta de risco (quantile forecasting)


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

X_train = splits["X_train"]
X_val = splits["X_val"]
y_train = splits["y_train"]

quantiles = [0.05, 0.25, 0.50, 0.75, 0.95]
preds = {}

for q in quantiles:
    model = GradientBoostingRegressor(
        loss="quantile",
        alpha=q,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    )
    model.fit(X_train, y_train)
    preds[q] = model.predict(X_val)

qf_dataset = pd.DataFrame(index=val_index)
qf_dataset["quantile_lower"] = preds[0.05]
qf_dataset["q25"] = preds[0.25]
qf_dataset["q50"] = preds[0.50]
qf_dataset["q75"] = preds[0.75]
qf_dataset["quantile_upper"] = preds[0.95]

qf_dataset["quantile_width"] = qf_dataset["quantile_upper"] - qf_dataset["quantile_lower"]
qf_dataset["quantile_skew"] = (
    (qf_dataset["q75"] - qf_dataset["q50"]) - (qf_dataset["q50"] - qf_dataset["q25"])
)

safe_width = qf_dataset["quantile_width"].replace(0, np.nan)
qf_dataset["asymmetry"] = (qf_dataset["q50"] - qf_dataset["quantile_lower"]) / safe_width
qf_dataset["asymmetry"] = qf_dataset["asymmetry"].replace([np.inf, -np.inf], np.nan).fillna(0.5)

qf_dataset = qf_dataset.sort_index()


## 4) Sanity checks mínimos


In [ ]:
print("NaN críticos:")
print(qf_dataset[["quantile_lower", "q50", "quantile_upper", "quantile_width"]].isna().sum())

print("\nDistribuição plausível:")
print(qf_dataset.describe().T[["mean", "std", "min", "max"]])

print("\nAlinhamento temporal:")
print("Index monotonic increasing:", qf_dataset.index.is_monotonic_increasing)
print("Index has duplicates:", qf_dataset.index.has_duplicates)

print("Dataset lines == val_index lines:", len(qf_dataset) == len(val_index))
print("Dataset index == val_index:", qf_dataset.index.equals(val_index))


## 5) DataFrame final (features de risco) e 6) Padronização


In [ ]:
qf_dataset = qf_dataset[[
    "quantile_lower",
    "q25",
    "q50",
    "q75",
    "quantile_upper",
    "quantile_width",
    "quantile_skew",
    "asymmetry",
]]

qf_dataset.head(), qf_dataset.shape


## 7) Salvamento (opcional)


In [ ]:
# qf_dataset.to_parquet("qf_dataset.parquet", index=True)
